## **Dự đoán tình trạng máy móc thiết bị công nghiệp bị khi đưa vào vận hành**.   

### **Data processing** 

In [1]:
import pandas as pd  
from sklearn.model_selection import train_test_split  
from lightgbm_classification import LightGBMClassification


In [2]:
# Đọc dữ liệu 
df = pd.read_csv (r'D:\machine-learning-group-6\classification\data\raw\machine_fail.csv') 
print ('Kích thước dataset', df.shape ) 

df.head ( 10 )

Kích thước dataset (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0
5,6,M14865,M,298.1,308.6,1425,41.9,11,0,0,0,0,0,0
6,7,L47186,L,298.1,308.6,1558,42.4,14,0,0,0,0,0,0
7,8,L47187,L,298.1,308.6,1527,40.2,16,0,0,0,0,0,0
8,9,M14868,M,298.3,308.7,1667,28.6,18,0,0,0,0,0,0
9,10,M14869,M,298.5,309.0,1741,28.0,21,0,0,0,0,0,0


In [3]:
# Kiểm tra kiểu dữ liệu từng feature  
df.dtypes  

UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

In [4]:
# Dữ liệu thiếu 
df.isnull ( ).sum ( )

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

In [5]:
# Chuẩn hóa dạng số của type of machine 
print ( df['Type'].value_counts ( ))   

df["Type"] = df["Type"].map({
    "L": 0,
    "M": 1,
    "H": 2
}) 

# Sau chuẩn hóa  
df.head ( )

Type
L    6000
M    2997
H    1003
Name: count, dtype: int64


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,1,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,0,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,0,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,0,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,0,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [6]:
# Chia dữ liệu thành 2 tập train và test 
X_features = df.drop(
    columns=[
        "Machine failure",
        "TWF",
        "HDF",
        "PWF",
        "OSF",
        "RNF",
        "UDI",
        "Product ID"
    ]
)

# Nhãn cần dự đoán
y_label = df["Machine failure"] 

# Chia 80% train - 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_label,
    test_size=0.2,
    random_state=42,
    stratify=y_label
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (8000, 6)
X_test: (2000, 6)
y_train: (8000,)
y_test: (2000,)


### **Training model** 

In [7]:
# Tiến hành gọi thuật toán xây dựng và training model 
model_lightGBM_cls = LightGBMClassification(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=15,
    max_depth=5
) 

print (LightGBMClassification)

<class 'lightgbm_classification.LightGBMClassification'>


In [8]:
# Training model  
model_lightGBM_cls.fit ( X_train , y_train )

In [9]:
# Predict trên tập test  
predict_test =  model_lightGBM_cls.predict (X_test)   
predict_test_proba = model_lightGBM_cls.predict_proba ( X_test ) 

# So sánh đánh giá tổng quan so với giá trị thực  
print ('Kết quả dự đoán với 10 thiết bị đầu tiên :', predict_test[:10] )  
print ('Kết quả thực tế của 10 thiết bị đầu tiên :' )  
print (y_test[:10])
print ('=== Kết quả dự đoán dưới dạng xác xuất ====')
print (predict_test_proba[:10])

Kết quả dự đoán với 10 thiết bị đầu tiên : [0 0 0 0 0 0 0 0 0 0]
Kết quả thực tế của 10 thiết bị đầu tiên :
2997    0
4871    0
3858    0
951     0
6463    0
3264    0
4508    0
2100    0
7885    0
2421    0
Name: Machine failure, dtype: int64
=== Kết quả dự đoán dưới dạng xác xuất ====
[[0.807625 0.192375]
 [0.999330 0.000670]
 [0.920699 0.079301]
 [0.998099 0.001901]
 [0.962462 0.037538]
 [0.998813 0.001187]
 [0.992586 0.007414]
 [0.998753 0.001247]
 [0.999714 0.000286]
 [0.899351 0.100649]]


###   **Model Performance Evaluation** 